In [1]:
import gymnasium as gym
import numpy as np
import math
import os
import configparser
from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy
from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.ppo_mask import MaskablePPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
import matplotlib.pyplot as plt
from sb3_contrib.common.maskable.utils import get_action_masks



In [2]:
from src.hpc_env import HPCenv
from src.validation import Validation
from src.baseline import PercentileBaseline
from src.utils import mask_fn, get_config_as_dict
from src.carbon_intensity import CarbonIntensity

In [3]:
# Use interactive widget backend if ipympl is available; otherwise fallback to inline
try:
    import ipympl  # noqa: F401
    get_ipython().run_line_magic('matplotlib', 'widget')
except Exception:
    get_ipython().run_line_magic('matplotlib', 'inline')
    print('ipympl not available; using inline Matplotlib backend. Install with `pip install ipympl` for widgets.')


ipympl not available; using inline Matplotlib backend. Install with `pip install ipympl` for widgets.


In [4]:
WORKLOAD_PATH = "data/workloads/lublin_256.swf"

# Load config with explicit path and typed parsing
config = configparser.ConfigParser()
config_path = os.path.join(os.getcwd(), 'config_file', 'config.ini')
config.read(config_path)

['/Users/mikkeldahl/green_scheduler_v2/config_file/config.ini']

## Model validation

In [5]:
val = Validation()
val.load_dir("results/converged_carbon_only")
processed_stats, raw_stats = val.run_baselines(n_eval_episodes=1, mode="validation")

run time mean:  3239.192658161937
run time std:  13380.071808903413
Max Allocated Processors: 128 ;max node: 256 ;max procs: 256 ;max execution time: 158506
Executing baseline:  FCFS Baseline
Episode  0


KeyboardInterrupt: 

In [8]:
val = Validation()
val.load_dir("results/CI_B16384_RC_LR-00001_ETA10.0_C-None_Lu")
val.validate_policy(n_eval_episodes=1, checkpoints=None, mode="validation")

run time mean:  3239.192658161937
run time std:  13380.071808903413
Max Allocated Processors: 128 ;max node: 256 ;max procs: 256 ;max execution time: 158506
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


{'seed_2_1000000_steps.zip': {'Avg Wait': np.float64(795.6917469409008),
  'Max Wait': np.float64(795.6917469409008),
  'Avg Response': np.float64(4034.8844051028377),
  'Avg Slowdown': np.float64(16.071375922082094),
  'Carbon Emissions': np.float64(4136070.292395036),
  'Weighted Carbon Emissions': np.float64(-4136070.292395036),
  'System Utilization': np.float64(0.2317026697216464),
  'Action Analysis': {'Total Actions': 36332,
   'Schedule Action Percentage': 21.14389518881427,
   'Fixed Delay Percentage': 74.43575911042608,
   'Wait Delay Percentage': 4.420345700759661,
   'Fixed Delays': {'300s': 26834, '3600s': 138, '86400s': 72},
   'Wait for Jobs': {'2 jobs': 1308, '1 jobs': 196, '3 jobs': 102}}},
 'seed_2_1500000_steps.zip': {'Avg Wait': np.float64(2266.7958864878938),
  'Max Wait': np.float64(2266.7958864878938),
  'Avg Response': np.float64(5505.988544649831),
  'Avg Slowdown': np.float64(55.29557153911262),
  'Carbon Emissions': np.float64(4147924.243340985),
  'Weighted 

In [ ]:
processed_stats

{'10-percentile Baseline': {'Avg Wait': np.float64(91309.37),
  'Max Wait': np.float64(91309.37),
  'Avg Response': np.float64(94677.36333333331),
  'Avg Slowdown': np.float64(9342.16265326278),
  'Carbon Emissions': np.float64(147243.24531520848),
  'Weighted Carbon Emissions': np.float64(-147243.24531520848),
  'System Utilization': np.float64(0.19626598470024278),
  'Action Analysis': {'Total Actions': 12470,
   'Schedule Action Percentage': 24.057738572574177,
   'Fixed Delay Percentage': 50.60144346431436,
   'Wait Delay Percentage': 25.340817963111466,
   'Fixed Delays': {'300s': 6310},
   'Wait for Jobs': {'1 jobs': 3160}}},
 '25-percentile Baseline': {'Avg Wait': np.float64(90285.97666666667),
  'Max Wait': np.float64(90285.97666666667),
  'Avg Response': np.float64(93653.97),
  'Avg Slowdown': np.float64(9238.322994151273),
  'Carbon Emissions': np.float64(146733.3470191898),
  'Weighted Carbon Emissions': np.float64(-146733.3470191898),
  'System Utilization': np.float64(0.19